| Model            | Input per timestep                          |
| ---------------- | ------------------------------------------- |
| Baseline         | Word2Vec word vector only                   |
| Melody Variant 1 | Word2Vec + global MIDI feature vector       |
| Melody Variant 2 | Word2Vec + time-aligned MIDI feature vector |


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.nn as nn
import torch.nn.functional as F
MAX_WORDS = 120
WORDS_PER_LINE = 8
END_TOKEN = "<EOS>"


In [14]:
# -------------------------
# 1. Reproducibility + Device
# -------------------------

import random
import numpy as np

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


In [15]:
# -------------------------
# 2. Special Tokens
# -------------------------

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"

SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN]

word2idx = {token: idx for idx, token in enumerate(SPECIAL_TOKENS)}
idx2word = {idx: token for token, idx in word2idx.items()}

PAD_IDX = word2idx[PAD_TOKEN]
UNK_IDX = word2idx[UNK_TOKEN]
SOS_IDX = word2idx[SOS_TOKEN]
EOS_IDX = word2idx[EOS_TOKEN]

print(word2idx)

{'<PAD>': 0, '<UNK>': 1, '<SOS>': 2, '<EOS>': 3}


In [16]:
# -------------------------
# 3. Model Hyperparameters
# -------------------------

WORD_DIM = 300
GLOBAL_MELODY_DIM = 7
ALIGNED_MELODY_DIM = 4

HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.3

BATCH_SIZE = 16
SEQ_LEN = 30
LEARNING_RATE = 1e-3
EPOCHS = 20

# LSTM model

In [ ]:
# -------------------------
# 4. LSTM Model
# -------------------------

class LyricsLSTM(nn.Module):
    def __init__(
        self,
        word_dim,
        melody_dim,
        hidden_dim,
        vocab_size,
        num_layers=2,
        dropout=0.3
    ):
        super().__init__()

        self.word_dim = word_dim
        self.melody_dim = melody_dim

        self.lstm = nn.LSTM(
            input_size=word_dim + melody_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, word_vectors, melody_features=None):
        if self.melody_dim > 0:
            if melody_features is None:
                raise ValueError("melody_features is required for this model")
            x = torch.cat([word_vectors, melody_features], dim=-1)
        else:
            x = word_vectors

        out, _ = self.lstm(x)
        out = self.dropout(out)
        logits = self.fc(out)

        return logits

#  Load Lyrics CSV


In [25]:
import pandas as pd

CSV_PATH = r"D:\Masters Study\2ndyear\Deep_Learning\DL-Assignment-3\Data\Archive (1)\lyrics_train_set.csv"

df = pd.read_csv(CSV_PATH, header=None)

df = df.iloc[:, :3]
df.columns = ["artist", "title", "lyrics"]

print(df.head())
print(df.columns)
print(df.shape)

           artist                           title  \
0      elton john              candle in the wind   
1  gerry rafferty                    baker street   
2  gerry rafferty             right down the line   
3     2 unlimited                    tribal dance   
4     2 unlimited  let the beat control your body   

                                              lyrics  
0  goodbye norma jean & though i never knew you a...  
1  winding your way down on baker street & lite i...  
2  you know i need your love & you've got that ho...  
3  come on check it out ya'll & (come on come on!...  
4  let the beat control your body & let the beat ...  
Index(['artist', 'title', 'lyrics'], dtype='object')
(600, 3)


In [26]:
# -------------------------
# 7. Text Cleaning
# -------------------------

import re

def clean_text(text):
    text = text.lower()

    # replace & with newline marker
    text = text.replace("&", " <NEWLINE> ")

    # remove unwanted characters
    text = re.sub(r"[^a-zA-Z0-9\s'<NEWLINE>]", "", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [27]:
df["clean_lyrics"] = df["lyrics"].apply(clean_text)

print(df["clean_lyrics"].iloc[0][:500])

goodbye norma jean <NEWLINE> though i never knew you at all <NEWLINE> you had the grace to hold yourself <NEWLINE> while those around you crawled <NEWLINE> they crawled out of the woodwork <NEWLINE> and they whispered into your brain <NEWLINE> they set you on the treadmill <NEWLINE> and they made you change your name <NEWLINE> and it seems to me you lived your life <NEWLINE> like a candle in the wind <NEWLINE> never knowing who to cling to <NEWLINE> when the rain set in <NEWLINE> and i would lik


In [28]:
# -------------------------
# 8. Tokenization
# -------------------------

def tokenize(text):
    return text.split()

df["tokens"] = df["clean_lyrics"].apply(tokenize)

print(df["tokens"].iloc[0][:30])

['goodbye', 'norma', 'jean', '<NEWLINE>', 'though', 'i', 'never', 'knew', 'you', 'at', 'all', '<NEWLINE>', 'you', 'had', 'the', 'grace', 'to', 'hold', 'yourself', '<NEWLINE>', 'while', 'those', 'around', 'you', 'crawled', '<NEWLINE>', 'they', 'crawled', 'out', 'of']


In [29]:
# -------------------------
# 9. Build Vocabulary
# -------------------------

from collections import Counter

word_counter = Counter()

for tokens in df["tokens"]:
    word_counter.update(tokens)

MIN_FREQ = 2

vocab_words = [
    word for word, count in word_counter.items()
    if count >= MIN_FREQ
]

print("Vocabulary size before filtering:", len(word_counter))
print("Vocabulary size after filtering:", len(vocab_words))

Vocabulary size before filtering: 7702
Vocabulary size after filtering: 4149


In [30]:
for word in vocab_words:
    if word not in word2idx:
        idx = len(word2idx)
        word2idx[word] = idx
        idx2word[idx] = word

vocab_size = len(word2idx)

print("Final vocab size:", vocab_size)

Final vocab size: 4153


In [31]:
# -------------------------
# 10. Convert Words to IDs
# -------------------------

def tokens_to_ids(tokens):
    ids = []

    for token in tokens:
        ids.append(word2idx.get(token, UNK_IDX))

    return ids

df["token_ids"] = df["tokens"].apply(tokens_to_ids)

print(df["token_ids"].iloc[0][:30])

[4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 7, 12, 15, 16, 17, 18, 19, 20, 7, 21, 22, 23, 12, 24, 7, 25, 24, 26, 27]
